# Sprint 4 - Rol 5: Experiment Tracker

Este notebook consolida el registro de experimentos del Sprint 4, integrando modelos tuneados, ensambles y modelo final.  
Se parte del `experiments_log.csv` generado en Sprint 3 y se actualiza con métricas reales calculadas sobre el conjunto de prueba.

**Objetivo del Rol 5:** garantizar trazabilidad, reproducibilidad y validación de los artefactos generados durante el proceso de modelado avanzado.


## 1. Flujo completo: de datos crudos a modelo final

```text
1. Datos crudos
   → data/raw/01-hotel_bookings.csv

2. Limpieza de datos
   → 05_data_cleaning.ipynb
   → data/interim/hotel_bookings_clean.csv

3. Feature Engineering
   → 06_feature_eng.ipynb
   → data/interim/hotel_bookings_fe.csv

4. Balanceo de clases
   → 07_class_balance.ipynb

5. Pipeline integrado
   → 08_pipeline.ipynb
   → models/preprocessor.pkl
   → data/processed/X_train_bal.csv
   → data/processed/X_test.csv
   → data/processed/y_train_bal.csv
   → data/processed/y_test.csv

6. Entrenamiento de modelos baseline
   → 09_baseline_models.ipynb
   → models/baseline_dt.pkl
   → models/baseline_rf.pkl
   → models/baseline_xgb.pkl
   → models/baseline_gb.pkl
   → models/baseline_lr.pkl
   → models/baseline_NN.pkl

7. Evaluación de modelos
   → 10_evaluation.ipynb

8. Comparación y selección de candidatos
   → 11_comparation.ipynb

9. Optimización de hiperparámetros
   → 12_hyperparam_tuning.ipynb
   → models/tuned_rf.pkl
   → models/tuned_xgb.pkl

10. Construcción de ensembles
    → 13_ensemble.ipynb
    → models/ensemble_hv.pkl
    → models/ensemble_sv.pkl
    → models/ensemble_stack.pkl

11. Validación final
    → 14_final_validation.ipynb
    → models/final_model.pkl

12. Experiment Tracking
    → 15_experiment_tracking.ipynb
    → models/experiments_log.csv
```


## 2. Importación de librerías

Se importan las librerías necesarias para cargar modelos, evaluar métricas, leer el registro de experimentos y actualizar el archivo `experiments_log.csv`.


In [23]:
import os
import sys
import joblib
import pandas as pd
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
)

# Permite importar módulos desde la raíz del proyecto si fuera necesario
sys.path.append(os.path.abspath(".."))

LOG_PATH = "../models/experiments_log.csv"
MODELS_DIR = "../models"


## 3. Carga de insumos

Se cargan los datos de prueba (`X_test`, `y_test`) y el archivo `experiments_log.csv` generado al cierre del Sprint 3.  
Este registro contiene los modelos baseline y será extendido con los resultados del Sprint 4.


In [24]:
# Cargar datos de prueba
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

# Cargar registro de experimentos existente
experiments_log = pd.read_csv(LOG_PATH)

print("Insumos cargados correctamente")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Registros actuales en experiments_log:", len(experiments_log))

experiments_log.head()


Insumos cargados correctamente
X_test shape: (23841, 64)
y_test shape: (23841,)
Registros actuales en experiments_log: 17


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_precision,...,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
0,dt,random_state=42,2026-05-05,0.816071,NaN,0.761476,0.754286,0.807445,0.821484,NaN,...,0.768712,0.820202,-0.011801,-0.014425,0.33,True,Rank 1 CV Recall. Sin overfitting. Seleccionad...,NaN,NaN,NaN
1,rf,random_state=42,2026-05-05,0.862808,NaN,0.752171,0.802576,0.926593,0.866994,NaN,...,0.815815,0.934606,-0.017034,-0.013239,4.33,True,Rank 2 CV Recall. Mejor AUC (0.927). Modelo má...,NaN,NaN,NaN
2,xgb,random_state=42,2026-05-05,0.846450,NaN,0.713676,0.775089,0.915453,0.845560,NaN,...,0.776850,0.916261,-0.002356,-0.001761,0.55,True,Rank 3 CV Recall. Gap mínimo de overfitting. E...,NaN,NaN,NaN
3,lr,"random_state=42, max_iter=1000",2026-05-05,0.812653,NaN,0.618413,0.709945,0.860963,0.784866,NaN,...,0.706379,0.863555,0.003298,0.003566,4.30,False,CV Recall bajo (0.618). Modelo lineal insufici...,NaN,NaN,NaN
4,gb,random_state=42,2026-05-05,0.818368,NaN,0.616122,0.715516,0.885453,0.814479,NaN,...,0.711231,0.883982,0.004628,0.004285,5.89,False,CV Recall bajo (0.616). Más lento que RF/XGB s...,NaN,NaN,NaN


## 4. Normalización de columnas del registro

Se verifica que `experiments_log.csv` tenga todas las columnas necesarias para registrar modelos baseline, modelos tuneados, ensambles y el modelo final.  
Si alguna columna no existe, se crea automáticamente.


In [25]:
required_columns = [
    "model", "params", "date",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc",
    "test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc",
    "recall_gap", "f1_gap", "train_time_s",
    "selected", "notes",
    "tipo_modelo", "dataset", "path"
]

for col in required_columns:
    if col not in experiments_log.columns:
        experiments_log[col] = None

# Mantener un orden estándar de columnas
experiments_log = experiments_log[required_columns]

# Guardar normalización
experiments_log.to_csv(LOG_PATH, index=False)

print("experiments_log.csv normalizado correctamente")
experiments_log.head()


experiments_log.csv normalizado correctamente


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_precision,...,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
0,dt,random_state=42,2026-05-05,0.816071,NaN,0.761476,0.754286,0.807445,0.821484,NaN,...,0.768712,0.820202,-0.011801,-0.014425,0.33,True,Rank 1 CV Recall. Sin overfitting. Seleccionad...,NaN,NaN,NaN
1,rf,random_state=42,2026-05-05,0.862808,NaN,0.752171,0.802576,0.926593,0.866994,NaN,...,0.815815,0.934606,-0.017034,-0.013239,4.33,True,Rank 2 CV Recall. Mejor AUC (0.927). Modelo má...,NaN,NaN,NaN
2,xgb,random_state=42,2026-05-05,0.846450,NaN,0.713676,0.775089,0.915453,0.845560,NaN,...,0.776850,0.916261,-0.002356,-0.001761,0.55,True,Rank 3 CV Recall. Gap mínimo de overfitting. E...,NaN,NaN,NaN
3,lr,"random_state=42, max_iter=1000",2026-05-05,0.812653,NaN,0.618413,0.709945,0.860963,0.784866,NaN,...,0.706379,0.863555,0.003298,0.003566,4.30,False,CV Recall bajo (0.618). Modelo lineal insufici...,NaN,NaN,NaN
4,gb,random_state=42,2026-05-05,0.818368,NaN,0.616122,0.715516,0.885453,0.814479,NaN,...,0.711231,0.883982,0.004628,0.004285,5.89,False,CV Recall bajo (0.616). Más lento que RF/XGB s...,NaN,NaN,NaN


## 5. Verificación de modelos y artefactos disponibles

Se identifican los archivos disponibles en la carpeta `models/`, incluyendo modelos `.pkl` y artefactos versionados con DVC (`.dvc`).  
Esta verificación permite confirmar qué modelos pueden cargarse directamente desde el entorno local.


In [26]:
artifacts = sorted([
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(".pkl") or f.endswith(".pkl.dvc") or f.endswith(".dvc")
])

print("Modelos y artefactos disponibles en /models:\n")

for f in artifacts:
    print("-", f)


Modelos y artefactos disponibles en /models:

- baseline_NN.pkl
- baseline_NN.pkl.dvc
- baseline_dt.pkl
- baseline_dt.pkl.dvc
- baseline_gb.pkl
- baseline_gb.pkl.dvc
- baseline_lr.pkl
- baseline_lr.pkl.dvc
- baseline_rf.pkl.dvc
- baseline_xgb.pkl
- baseline_xgb.pkl.dvc
- ensemble_hv.pkl
- ensemble_hv.pkl.dvc
- ensemble_stack.pkl
- ensemble_stack.pkl.dvc
- ensemble_sv.pkl
- ensemble_sv.pkl.dvc
- final_model.pkl
- preprocessor.pkl
- preprocessor.pkl.dvc
- tuned_rf.pkl
- tuned_rf.pkl.dvc
- tuned_xgb.pkl
- tuned_xgb.pkl.dvc


## 6. Funciones auxiliares

Se definen funciones para:

- Identificar el tipo de modelo según el nombre del archivo.
- Recuperar parámetros del modelo.
- Evaluar métricas en el conjunto de prueba.
- Registrar nuevos experimentos sin borrar registros previos.


In [27]:
def infer_tipo_modelo(filename):
    """Identifica el tipo de modelo según el nombre del archivo."""
    if filename.startswith("tuned_"):
        return "tuned"
    elif filename.startswith("ensemble_"):
        return "ensemble"
    elif filename == "final_model.pkl":
        return "final"
    elif filename.startswith("baseline_"):
        return "baseline"
    else:
        return "unknown"


def infer_model_name(filename):
    """Limpia el nombre del archivo para obtener el nombre del modelo."""
    name = filename.replace(".pkl", "")
    name = name.replace("tuned_", "")
    name = name.replace("ensemble_", "")
    name = name.replace("baseline_", "")
    return name


def get_model_params(model):
    """Intenta recuperar los parámetros del modelo."""
    try:
        return str(model.get_params())
    except Exception:
        return "Parámetros no disponibles"


def evaluate_model(model, X_test, y_test):
    """Evalúa un modelo sobre el conjunto de prueba."""
    y_pred = model.predict(X_test)

    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred, zero_division=0),
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_roc_auc": None
    }

    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_test)[:, 1]
            metrics["test_roc_auc"] = roc_auc_score(y_test, y_proba)
        except Exception as e:
            print("No se pudo calcular ROC-AUC:", e)

    return metrics


def save_experiment(result, log_path=LOG_PATH):
    """Agrega un experimento nuevo al registro sin borrar los anteriores."""
    df_new = pd.DataFrame([result])

    if os.path.exists(log_path):
        df_old = pd.read_csv(log_path)

        # Asegurar compatibilidad de columnas
        for col in df_new.columns:
            if col not in df_old.columns:
                df_old[col] = None

        for col in df_old.columns:
            if col not in df_new.columns:
                df_new[col] = None

        df_new = df_new[df_old.columns]

        # Evitar duplicados por path
        if "path" in df_old.columns and str(result["path"]) in df_old["path"].astype(str).values:
            print(f" El modelo ya estaba registrado: {result['path']}")
            return df_old

        df_final = pd.concat([df_old, df_new], ignore_index=True)

    else:
        df_final = df_new

    df_final.to_csv(log_path, index=False)

    print(" Experimento registrado correctamente")
    print("Total registros:", len(df_final))

    return df_final


## 7. Limpieza de registros incompletos

Se eliminan registros generados como plantilla que contengan valores temporales, por ejemplo `REEMPLAZAR_CON...` o métricas en cero.  
Esto evita que el log final conserve filas incompletas del proceso de desarrollo.


In [28]:
# Eliminar registros incompletos generados como plantilla
mask_incomplete = (
    experiments_log["params"].astype(str).str.contains("REEMPLAZAR", na=False) |
    (
        experiments_log["tipo_modelo"].isin(["tuned", "ensemble", "final"]) &
        (experiments_log["test_f1"].fillna(0) == 0)
    )
)

removed = experiments_log[mask_incomplete]
experiments_log = experiments_log[~mask_incomplete].reset_index(drop=True)

experiments_log.to_csv(LOG_PATH, index=False)

print("Registros incompletos eliminados:", len(removed))
experiments_log.tail()


Registros incompletos eliminados: 0


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_precision,...,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
12,xgb,"{'memory': None, 'steps': [('clf', XGBClassifi...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.867371,0.853531,...,0.812544,0.935699,NaN,NaN,NaN,False,Registro automático Sprint 4 desde tuned_xgb.pkl,tuned,global,../models\tuned_xgb.pkl
13,hv,"{'estimators': [('rf', Pipeline(steps=[('clf',...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.864645,0.857817,...,0.806546,NaN,NaN,NaN,NaN,False,Registro automático Sprint 4 desde ensemble_hv...,ensemble,global,../models\ensemble_hv.pkl
14,sv,"{'estimators': [('rf', Pipeline(steps=[('clf',...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.866784,0.830667,...,0.817492,0.935269,NaN,NaN,NaN,False,Registro automático Sprint 4 desde ensemble_sv...,ensemble,global,../models\ensemble_sv.pkl
15,stack,"{'cv': 5, 'estimators': [('xgb', Pipeline(step...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.866239,0.822562,...,0.818776,0.935132,NaN,NaN,NaN,False,Registro automático Sprint 4 desde ensemble_st...,ensemble,global,../models\ensemble_stack.pkl
16,final_model,"{'memory': None, 'steps': [('clf', XGBClassifi...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.867371,0.853531,...,0.812544,0.935699,NaN,NaN,NaN,True,Registro automático Sprint 4 desde final_model...,final,global,../models\final_model.pkl


## 8. Registro automático de modelos del Sprint 4

Se cargan los modelos reales disponibles en `models/` y se calculan sus métricas sobre el conjunto de prueba.  
Los modelos esperados son:

- `tuned_rf.pkl`
- `tuned_xgb.pkl`
- `ensemble_hv.pkl`
- `ensemble_sv.pkl`
- `ensemble_stack.pkl`
- `final_model.pkl`


In [29]:
model_files_to_register = [
    "tuned_rf.pkl",
    "tuned_xgb.pkl",
    "ensemble_hv.pkl",
    "ensemble_sv.pkl",
    "ensemble_stack.pkl",
    "final_model.pkl"
]

new_results = []

for filename in model_files_to_register:
    path = os.path.join(MODELS_DIR, filename)

    if not os.path.exists(path):
        print(f" No se encontró {filename}")
        continue

    # Cargar modelo real
    model = joblib.load(path)

    # Evaluar modelo sobre X_test
    metrics = evaluate_model(model, X_test, y_test)

    tipo_modelo = infer_tipo_modelo(filename)
    model_name = infer_model_name(filename)

    result = {
        "model": model_name,
        "params": get_model_params(model),
        "date": datetime.today().strftime("%Y-%m-%d"),

        # Métricas CV no se recalculan aquí; pertenecen a notebooks de tuning/ensamble
        "cv_accuracy": None,
        "cv_precision": None,
        "cv_recall": None,
        "cv_f1": None,
        "cv_roc_auc": None,

        # Métricas calculadas en test
        "test_accuracy": round(metrics["test_accuracy"], 6),
        "test_precision": round(metrics["test_precision"], 6),
        "test_recall": round(metrics["test_recall"], 6),
        "test_f1": round(metrics["test_f1"], 6),
        "test_roc_auc": round(metrics["test_roc_auc"], 6) if metrics["test_roc_auc"] is not None else None,

        # Gaps no se calculan porque las métricas CV no se recalculan en este notebook
        "recall_gap": None,
        "f1_gap": None,
        "train_time_s": None,

        "selected": True if filename == "final_model.pkl" else False,
        "notes": f"Registro automático Sprint 4 desde {filename}",

        "tipo_modelo": tipo_modelo,
        "dataset": "global",
        "path": path
    }

    new_results.append(result)

print(" Modelos registrados desde archivos reales:", len(new_results))

pd.DataFrame(new_results)


 Modelos registrados desde archivos reales: 6


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_precision,...,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
0,rf,"{'memory': None, 'steps': [('clf', RandomFores...",2026-05-08,None,None,None,None,None,0.860115,0.877193,...,0.793306,0.930729,None,None,None,False,Registro automático Sprint 4 desde tuned_rf.pkl,tuned,global,../models\tuned_rf.pkl
1,xgb,"{'memory': None, 'steps': [('clf', XGBClassifi...",2026-05-08,None,None,None,None,None,0.867371,0.853531,...,0.812544,0.935699,None,None,None,False,Registro automático Sprint 4 desde tuned_xgb.pkl,tuned,global,../models\tuned_xgb.pkl
2,hv,"{'estimators': [('rf', Pipeline(steps=[('clf',...",2026-05-08,None,None,None,None,None,0.864645,0.857817,...,0.806546,NaN,None,None,None,False,Registro automático Sprint 4 desde ensemble_hv...,ensemble,global,../models\ensemble_hv.pkl
3,sv,"{'estimators': [('rf', Pipeline(steps=[('clf',...",2026-05-08,None,None,None,None,None,0.866784,0.830667,...,0.817492,0.935269,None,None,None,False,Registro automático Sprint 4 desde ensemble_sv...,ensemble,global,../models\ensemble_sv.pkl
4,stack,"{'cv': 5, 'estimators': [('xgb', Pipeline(step...",2026-05-08,None,None,None,None,None,0.866239,0.822562,...,0.818776,0.935132,None,None,None,False,Registro automático Sprint 4 desde ensemble_st...,ensemble,global,../models\ensemble_stack.pkl
5,final_model,"{'memory': None, 'steps': [('clf', XGBClassifi...",2026-05-08,None,None,None,None,None,0.867371,0.853531,...,0.812544,0.935699,None,None,None,True,Registro automático Sprint 4 desde final_model...,final,global,../models\final_model.pkl


## 9. Actualización de `experiments_log.csv`

Se agregan los resultados del Sprint 4 al registro histórico, conservando los registros del Sprint 3.  
El archivo se actualiza sin duplicar modelos ya registrados por ruta (`path`).


In [30]:
new_results_df = pd.DataFrame(new_results)

if len(new_results_df) > 0:
    existing_paths = experiments_log["path"].dropna().astype(str).tolist()

    # Evitar duplicados por path
    new_results_df = new_results_df[
        ~new_results_df["path"].astype(str).isin(existing_paths)
    ]

    experiments_log_updated = pd.concat(
        [experiments_log, new_results_df],
        ignore_index=True
    )

    experiments_log_updated.to_csv(LOG_PATH, index=False)

    print(" experiments_log.csv actualizado correctamente")
    print("Total registros:", len(experiments_log_updated))
else:
    experiments_log_updated = experiments_log
    print(" No se agregaron nuevos registros")

experiments_log_updated.tail(10)


 experiments_log.csv actualizado correctamente
Total registros: 17


,model,params,date,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_precision,...,test_f1,test_roc_auc,recall_gap,f1_gap,train_time_s,selected,notes,tipo_modelo,dataset,path
7,tuned_xgb,"n_estimators=500, max_depth=5, learning_rate=0...",2026-05-08,NaN,NaN,0.7663,0.8041,0.9273,NaN,NaN,...,0.812500,0.935700,-0.0091,-0.0084,NaN,True,XGB tuneado con RandomizedSearchCV. MODELO FIN...,NaN,NaN,NaN
8,ensemble_hv,"voting=hard, estimators=[tuned_rf, tuned_xgb]",2026-05-08,NaN,NaN,0.6923,0.7806,NaN,NaN,NaN,...,0.792900,NaN,-0.016,-0.0123,NaN,False,"Hard Voting. No soporta predict_proba, AUC no ...",NaN,NaN,NaN
9,ensemble_sv,"voting=soft, weights=[3,3], estimators=[tuned_...",2026-05-08,NaN,NaN,0.7495,0.8026,0.9299,NaN,NaN,...,0.812800,0.937000,-0.0108,-0.0102,NaN,False,Soft Voting con pesos iguales. AUC más alto de...,NaN,NaN,NaN
10,ensemble_stack,"estimators=[tuned_xgb, tuned_rf], final_estima...",2026-05-08,NaN,NaN,0.7619,0.8054,0.9293,NaN,NaN,...,0.815000,0.936400,-0.0126,-0.0096,NaN,False,Stacking con meta-learner LR. F1 más alto pero...,NaN,NaN,NaN
11,rf,"{'memory': None, 'steps': [('clf', RandomFores...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.860115,0.877193,...,0.793306,0.930729,NaN,NaN,NaN,False,Registro automático Sprint 4 desde tuned_rf.pkl,tuned,global,../models\tuned_rf.pkl
12,xgb,"{'memory': None, 'steps': [('clf', XGBClassifi...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.867371,0.853531,...,0.812544,0.935699,NaN,NaN,NaN,False,Registro automático Sprint 4 desde tuned_xgb.pkl,tuned,global,../models\tuned_xgb.pkl
13,hv,"{'estimators': [('rf', Pipeline(steps=[('clf',...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.864645,0.857817,...,0.806546,NaN,NaN,NaN,NaN,False,Registro automático Sprint 4 desde ensemble_hv...,ensemble,global,../models\ensemble_hv.pkl
14,sv,"{'estimators': [('rf', Pipeline(steps=[('clf',...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.866784,0.830667,...,0.817492,0.935269,NaN,NaN,NaN,False,Registro automático Sprint 4 desde ensemble_sv...,ensemble,global,../models\ensemble_sv.pkl
15,stack,"{'cv': 5, 'estimators': [('xgb', Pipeline(step...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.866239,0.822562,...,0.818776,0.935132,NaN,NaN,NaN,False,Registro automático Sprint 4 desde ensemble_st...,ensemble,global,../models\ensemble_stack.pkl
16,final_model,"{'memory': None, 'steps': [('clf', XGBClassifi...",2026-05-08,NaN,NaN,NaN,NaN,NaN,0.867371,0.853531,...,0.812544,0.935699,NaN,NaN,NaN,True,Registro automático Sprint 4 desde final_model...,final,global,../models\final_model.pkl


## 10. Validación del modelo final

Se valida que `final_model.pkl` pueda cargarse correctamente y generar predicciones sobre una muestra de `X_test`.  
Esta validación confirma que el artefacto final puede reutilizarse.


In [31]:
final_model_path = os.path.join(MODELS_DIR, "final_model.pkl")

if os.path.exists(final_model_path):
    final_model = joblib.load(final_model_path)
    pred = final_model.predict(X_test.head())

    print(" final_model.pkl carga correctamente")
    print("Predicciones de prueba:", pred)
else:
    print(" No existe final_model.pkl")


 final_model.pkl carga correctamente
Predicciones de prueba: [1 0 0 1 1]


## 11. Validación del módulo `tuning.py`

Se verifica que el módulo `src/tuning.py` pueda importarse correctamente y que las funciones principales del sistema de tuning estén disponibles para su reutilización dentro del pipeline del Sprint 4.

In [32]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.tuning import (
    tune_model,
    save_experiment,
    validate_model_load
)

print(" tuning.py importado correctamente")
print("Funciones disponibles:")
print("- tune_model")
print("- save_experiment")
print("- validate_model_load")

 tuning.py importado correctamente
Funciones disponibles:
- tune_model
- save_experiment
- validate_model_load


## 12. Resumen final del registro

Se muestra una vista resumida del registro actualizado, incluyendo modelos baseline, tuneados, ensambles y modelo final.


In [33]:
experiments_log_updated = pd.read_csv(LOG_PATH)

cols_show = [
    "model", "tipo_modelo", "dataset",
    "test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc",
    "selected", "path", "notes"
]

available_cols = [col for col in cols_show if col in experiments_log_updated.columns]

experiments_log_updated[available_cols]


,model,tipo_modelo,dataset,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,selected,path,notes
0,dt,NaN,NaN,0.821484,NaN,0.773278,0.768712,0.820202,True,NaN,Rank 1 CV Recall. Sin overfitting. Seleccionad...
1,rf,NaN,NaN,0.866994,NaN,0.769205,0.815815,0.934606,True,NaN,Rank 2 CV Recall. Mejor AUC (0.927). Modelo má...
2,xgb,NaN,NaN,0.845560,NaN,0.716031,0.776850,0.916261,True,NaN,Rank 3 CV Recall. Gap mínimo de overfitting. E...
3,lr,NaN,NaN,0.784866,NaN,0.615115,0.706379,0.863555,False,NaN,CV Recall bajo (0.618). Modelo lineal insufici...
4,gb,NaN,NaN,0.814479,NaN,0.611495,0.711231,0.883982,False,NaN,CV Recall bajo (0.616). Más lento que RF/XGB s...
5,NN,NaN,NaN,NaN,NaN,0.647585,0.726027,0.885184,False,NaN,CV Recall más bajo (0.615). Mayor tiempo de en...
6,tuned_rf,NaN,NaN,NaN,NaN,0.724100,0.793300,0.930700,False,NaN,RF tuneado con RandomizedSearchCV. Mejora reca...
7,tuned_xgb,NaN,NaN,NaN,NaN,0.775300,0.812500,0.935700,True,NaN,XGB tuneado con RandomizedSearchCV. MODELO FIN...
8,ensemble_hv,NaN,NaN,NaN,NaN,0.708300,0.792900,NaN,False,NaN,"Hard Voting. No soporta predict_proba, AUC no ..."
9,ensemble_sv,NaN,NaN,NaN,NaN,0.760300,0.812800,0.937000,False,NaN,Soft Voting con pesos iguales. AUC más alto de...


## 13. Vista específica del Sprint 4

Se filtran únicamente los modelos generados durante el Sprint 4: modelos tuneados, ensambles y modelo final.


In [34]:
sprint4_log = experiments_log_updated[
    experiments_log_updated["tipo_modelo"].isin(["tuned", "ensemble", "final"])
]

sprint4_log[available_cols]


,model,tipo_modelo,dataset,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,selected,path,notes
11,rf,tuned,global,0.860115,0.877193,0.724064,0.793306,0.930729,False,../models\tuned_rf.pkl,Registro automático Sprint 4 desde tuned_rf.pkl
12,xgb,tuned,global,0.867371,0.853531,0.775314,0.812544,0.935699,False,../models\tuned_xgb.pkl,Registro automático Sprint 4 desde tuned_xgb.pkl
13,hv,ensemble,global,0.864645,0.857817,0.761059,0.806546,NaN,False,../models\ensemble_hv.pkl,Registro automático Sprint 4 desde ensemble_hv...
14,sv,ensemble,global,0.866784,0.830667,0.804729,0.817492,0.935269,False,../models\ensemble_sv.pkl,Registro automático Sprint 4 desde ensemble_sv...
15,stack,ensemble,global,0.866239,0.822562,0.815024,0.818776,0.935132,False,../models\ensemble_stack.pkl,Registro automático Sprint 4 desde ensemble_st...
16,final_model,final,global,0.867371,0.853531,0.775314,0.812544,0.935699,True,../models\final_model.pkl,Registro automático Sprint 4 desde final_model...


## 13. Conclusión

Se actualizó el archivo `experiments_log.csv` incorporando el modelo final validado del Sprint 4, manteniendo la trazabilidad de los experimentos desde los modelos baseline del Sprint 3 hasta el modelo optimizado seleccionado.

Asimismo, se verificó la correcta integración del módulo `src/tuning.py`, el cual centraliza las utilidades de búsqueda de hiperparámetros, persistencia de modelos y validación de carga de artefactos reutilizables dentro del pipeline de Machine Learning.

El archivo `final_model.pkl` fue cargado exitosamente mediante `joblib` y generó predicciones sobre el conjunto de prueba, confirmando la reutilización del modelo final en escenarios posteriores de inferencia.

Finalmente, el proyecto consolida un flujo reproducible que integra entrenamiento, tuning, versionado de artefactos, experiment tracking y documentación técnica para el cierre del Sprint 4.
